<a href="https://colab.research.google.com/github/naisyanabilapratiwi14-droid/Tugas-4-Sistem-temu-kembali/blob/main/240210500003_Naisya_Nabila_Pratiwi_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

No. 2 — Pembobotan (Term Weighting)
### Bag-of-Words, TF, IDF, dan TF-IDF

**Studi Kasus:** Prototipe sistem pencarian dokumen teknis untuk mahasiswa Teknik Komputer.
Dokumen-dokumen singkat diubah menjadi representasi numerik menggunakan TF-IDF agar dapat
digunakan untuk perhitungan kemiripan (*cosine similarity*).


In [2]:
!pip install Sastrawi -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 2.3 MB/s eta 0:00:00


In [3]:
import re # Import "re" untuk operasi regular expression (dipakai pada tahap cleaning)
import math # Import "math" untuk fungsi logaritma natural (ln), dipakai pada perhitungan IDF manual
import pandas as pd # Import "pandas" untuk menampilkan hasil dalam bentuk tabel (DataFrame)


from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer dari scikit-learn untuk perhitungan TF-IDF menggunakan library

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Import factory pembuat stopword remover dari library Sastrawi (untuk Bahasa Indonesia)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
stopwords_list = set(StopWordRemoverFactory().get_stop_words()) # Buat daftar stopwords Bahasa Indonesia dan simpan sebagai set (pencarian lebih cepat)

pd.set_option("display.max_columns", None) # Atur agar seluruh kolom pandas DataFrame ditampilkan (tidak terpotong)
pd.set_option("display.width", 120) # Atur lebar tampilan output pandas agar tabel lebih mudah dibaca

In [4]:
# List berisi 4 dokumen pendek (mentah, belum diproses)
documents = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang saling terhubung.",   # D1: sistem komputer
    "Jaringan komputer memungkinkan pertukaran data antar perangkat secara cepat dan efisien.",   # D2: jaringan komputer
    "Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas seperti manusia.",  # D3: AI
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dari kumpulan data yang besar.",  # D4: IR
]

# Cetak tiap dokumen beserta nomornya sebagai pengecekan awal
for i, doc in enumerate(documents, 1):     # enumerate mulai dari 1 agar penomoran D1..D4 natural
    print(f"D{i}: {doc}")                  # cetak nomor dan isi dokumen

D1: Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang saling terhubung.
D2: Jaringan komputer memungkinkan pertukaran data antar perangkat secara cepat dan efisien.
D3: Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas seperti manusia.
D4: Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dari kumpulan data yang besar.


In [5]:
def simple_preprocess(text):
    """Preprocessing sederhana: case folding -> cleaning -> tokenisasi -> stopwords removal."""
    text = text.lower()                                     # case folding: ubah ke huruf kecil semua
    text = re.sub(r"[^a-z\s]", " ", text)                    # cleaning: hapus tanda baca & angka
    text = re.sub(r"\s+", " ", text).strip()                 # rapikan spasi ganda jadi satu spasi
    tokens = text.split()                                    # tokenisasi: pecah jadi list kata
    tokens = [t for t in tokens if t not in stopwords_list]  # stopwords removal: buang kata umum
    return tokens                                            # kembalikan list token bersih


# Terapkan preprocessing sederhana ke seluruh dokumen, hasil disimpan sebagai list of list token
tokenized_docs = [simple_preprocess(doc) for doc in documents]

# Cetak hasil tokenisasi tiap dokumen sebagai pengecekan
for i, tokens in enumerate(tokenized_docs, 1):               # loop tiap dokumen, nomor mulai dari 1
    print(f"D{i} ({len(tokens)} token):", tokens)            # cetak jumlah token & isi tokennya

D1 (9 token): ['sistem', 'komputer', 'terdiri', 'perangkat', 'keras', 'perangkat', 'lunak', 'saling', 'terhubung']
D2 (9 token): ['jaringan', 'komputer', 'memungkinkan', 'pertukaran', 'data', 'antar', 'perangkat', 'cepat', 'efisien']
D3 (11 token): ['kecerdasan', 'buatan', 'cabang', 'ilmu', 'komputer', 'mempelajari', 'cara', 'membuat', 'mesin', 'cerdas', 'manusia']
D4 (11 token): ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan', 'kumpulan', 'data', 'besar']


In [6]:
# Bangun vocabulary: kumpulan kata unik dari SELURUH dokumen, diurutkan alfabetis
vocabulary = sorted(set(word for tokens in tokenized_docs for word in tokens))

print("Ukuran vocabulary:", len(vocabulary))   # cetak jumlah kata unik
print(vocabulary)                              # cetak daftar kata dalam vocabulary

# Bangun matriks Bag-of-Words (raw count): baris = dokumen, kolom = term dari vocabulary
bow_data = []                                              # list kosong untuk menampung tiap baris matriks
for tokens in tokenized_docs:                              # loop untuk setiap dokumen (list token-nya)
    row = [tokens.count(term) for term in vocabulary]      # hitung kemunculan tiap term di dokumen ini
    bow_data.append(row)                                    # tambahkan baris hasil ke bow_data

# Ubah bow_data menjadi DataFrame pandas agar mudah dibaca sebagai tabel
df_bow = pd.DataFrame(
    bow_data,                                               # data matriks (raw count)
    columns=vocabulary,                                     # nama kolom = daftar term
    index=[f"D{i+1}" for i in range(len(documents))],       # nama baris = D1, D2, D3, D4
)

df_bow   # tampilkan tabel Bag-of-Words

Ukuran vocabulary: 34
['antar', 'besar', 'buatan', 'cabang', 'cara', 'cepat', 'cerdas', 'data', 'dokumen', 'efisien', 'ilmu', 'informasi', 'jaringan', 'kecerdasan', 'keras', 'komputer', 'kumpulan', 'lunak', 'manusia', 'membantu', 'membuat', 'mempelajari', 'memungkinkan', 'menemukan', 'mesin', 'pengguna', 'perangkat', 'pertukaran', 'relevan', 'saling', 'sistem', 'temu', 'terdiri', 'terhubung']


,antar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,efisien,ilmu,informasi,jaringan,kecerdasan,keras,komputer,kumpulan,lunak,manusia,membantu,membuat,mempelajari,memungkinkan,menemukan,mesin,pengguna,perangkat,pertukaran,relevan,saling,sistem,temu,terdiri,terhubung
D1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,2,0,0,1,1,0,1,1
D2,1,0,0,0,0,1,0,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0
D3,0,0,1,1,1,0,1,0,0,0,1,0,0,1,0,1,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0
D4,0,1,0,0,0,0,0,1,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,1,0,1,1,0,0


In [7]:
# Term Frequency (TF) = raw count, sehingga cukup menyalin matriks Bag-of-Words
df_tf = df_bow.copy()          # copy() dipakai agar df_bow asli tidak ikut berubah saat df_tf diubah nanti

print("=== Term Frequency (TF) ===")
df_tf   # tampilkan tabel TF

=== Term Frequency (TF) ===


,antar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,efisien,ilmu,informasi,jaringan,kecerdasan,keras,komputer,kumpulan,lunak,manusia,membantu,membuat,mempelajari,memungkinkan,menemukan,mesin,pengguna,perangkat,pertukaran,relevan,saling,sistem,temu,terdiri,terhubung
D1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,2,0,0,1,1,0,1,1
D2,1,0,0,0,0,1,0,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0
D3,0,0,1,1,1,0,1,0,0,0,1,0,0,1,0,1,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0
D4,0,1,0,0,0,0,0,1,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,1,0,1,1,0,0


In [8]:
N = len(documents)   # N = jumlah total dokumen (di sini N = 4)

# Hitung Document Frequency (DF): untuk tiap term, hitung berapa dokumen yang mengandungnya (nilai BoW > 0)
df_values = {term: int((df_bow[term] > 0).sum()) for term in vocabulary}

# Hitung Inverse Document Frequency (IDF) memakai formula: ln(N / df) + 1
idf_values = {term: math.log(N / df_values[term]) + 1 for term in vocabulary}

# Susun DF dan IDF ke dalam satu tabel pandas agar mudah dibaca
df_idf = pd.DataFrame({
    "Term": vocabulary,                                            # kolom nama term
    "DF": [df_values[t] for t in vocabulary],                      # kolom nilai DF tiap term
    "IDF (ln(N/df)+1)": [round(idf_values[t], 4) for t in vocabulary],  # kolom nilai IDF (dibulatkan)
}).set_index("Term")                                                # jadikan kolom "Term" sebagai index tabel

print(f"N (jumlah dokumen) = {N}\n")   # cetak jumlah dokumen sebagai konteks
df_idf                                  # tampilkan tabel DF & IDF

N (jumlah dokumen) = 4



,DF,IDF (ln(N/df)+1)
Term,,
antar,1,2.3863
besar,1,2.3863
buatan,1,2.3863
cabang,1,2.3863
cara,1,2.3863
cepat,1,2.3863
cerdas,1,2.3863
data,2,1.6931
dokumen,1,2.3863


In [9]:
# Salin df_tf sebagai basis, lalu ubah tipe datanya menjadi float (karena hasil TF-IDF berupa desimal)
df_tfidf_manual = df_tf.copy().astype(float)

# Kalikan setiap kolom TF dengan nilai IDF term yang bersesuaian
for term in vocabulary:                              # loop untuk setiap term dalam vocabulary
    df_tfidf_manual[term] = df_tf[term] * idf_values[term]   # TF-IDF = TF x IDF

df_tfidf_manual = df_tfidf_manual.round(4)   # bulatkan hasil ke 4 angka desimal agar rapi

print("=== TF-IDF (Manual) ===")
df_tfidf_manual   # tampilkan tabel TF-IDF hasil perhitungan manual

=== TF-IDF (Manual) ===


,antar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,efisien,ilmu,informasi,jaringan,kecerdasan,keras,komputer,kumpulan,lunak,manusia,membantu,membuat,mempelajari,memungkinkan,menemukan,mesin,pengguna,perangkat,pertukaran,relevan,saling,sistem,temu,terdiri,terhubung
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,1.2877,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,3.3863,0.0000,0.0000,2.3863,1.6931,0.0000,2.3863,2.3863
D2,2.3863,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,1.6931,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,1.2877,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,1.6931,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,2.3863,2.3863,2.3863,0.0000,2.3863,0.0000,0.0000,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,1.2877,0.0000,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,1.6931,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,2.3863,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,1.6931,2.3863,0.0000,0.0000


In [10]:
# Gabungkan kembali token hasil preprocessing menjadi satu string per dokumen
# (karena TfidfVectorizer butuh input berupa string, bukan list token)
preprocessed_texts = [" ".join(tokens) for tokens in tokenized_docs]

# Buat objek TfidfVectorizer dengan pengaturan agar formula IDF sama dengan perhitungan manual
vectorizer = TfidfVectorizer(
    norm=None,          # matikan normalisasi L2 agar nilainya bisa dibandingkan langsung dengan manual
    smooth_idf=False,   # gunakan formula IDF standar: ln(N/df) + 1 (tanpa smoothing tambahan)
    use_idf=True,        # aktifkan pembobotan IDF (bukan hanya TF)
)

# fit_transform: pelajari vocabulary dari data, sekaligus hitung matriks TF-IDF-nya
tfidf_matrix = vectorizer.fit_transform(preprocessed_texts)

# Ubah hasil (sparse matrix) menjadi DataFrame pandas agar mudah dibaca
df_tfidf_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),                        # ubah sparse matrix menjadi array biasa
    columns=vectorizer.get_feature_names_out(),      # nama kolom = daftar term hasil vectorizer
    index=[f"D{i+1}" for i in range(len(documents))],  # nama baris = D1..D4
)

# Urutkan ulang kolom agar urutannya SAMA dengan vocabulary manual (memudahkan perbandingan)
df_tfidf_sklearn = df_tfidf_sklearn[vocabulary].round(4)

print("=== TF-IDF (scikit-learn, norm=None, smooth_idf=False) ===")
print(df_tfidf_sklearn)   # cetak tabel TF-IDF hasil scikit-learn

# Hitung selisih absolut antara hasil manual dan hasil scikit-learn
selisih = (df_tfidf_manual - df_tfidf_sklearn).abs()

print("\nSelisih absolut maksimum antara hasil manual dan scikit-learn:", selisih.values.max())
print("=> Nilai selisih mendekati 0 berarti perhitungan manual SESUAI dengan hasil scikit-learn.\n")

selisih   # tampilkan tabel selisih (seharusnya seluruh nilainya 0 atau sangat mendekati 0)

=== TF-IDF (scikit-learn, norm=None, smooth_idf=False) ===
     antar   besar  buatan  cabang    cara   cepat  cerdas    data  dokumen  efisien    ilmu  informasi  jaringan  \
D1  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000   0.0000   0.0000  0.0000     0.0000    0.0000   
D2  2.3863  0.0000  0.0000  0.0000  0.0000  2.3863  0.0000  1.6931   0.0000   2.3863  0.0000     0.0000    2.3863   
D3  0.0000  0.0000  2.3863  2.3863  2.3863  0.0000  2.3863  0.0000   0.0000   0.0000  2.3863     0.0000    0.0000   
D4  0.0000  2.3863  0.0000  0.0000  0.0000  0.0000  0.0000  1.6931   2.3863   0.0000  0.0000     2.3863    0.0000   

    kecerdasan   keras  komputer  kumpulan   lunak  manusia  membantu  membuat  mempelajari  memungkinkan  menemukan  \
D1      0.0000  2.3863    1.2877    0.0000  2.3863   0.0000    0.0000   0.0000       0.0000        0.0000     0.0000   
D2      0.0000  0.0000    1.2877    0.0000  0.0000   0.0000    0.0000   0.0000       0.0000        2.3863     0.000

,antar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,efisien,ilmu,informasi,jaringan,kecerdasan,keras,komputer,kumpulan,lunak,manusia,membantu,membuat,mempelajari,memungkinkan,menemukan,mesin,pengguna,perangkat,pertukaran,relevan,saling,sistem,temu,terdiri,terhubung
D1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Buat vectorizer dengan pengaturan DEFAULT (smooth_idf=True, norm="l2")
vectorizer_default = TfidfVectorizer()   # tanpa parameter khusus = pengaturan bawaan scikit-learn

# Hitung matriks TF-IDF dengan pengaturan default tersebut
tfidf_default = vectorizer_default.fit_transform(preprocessed_texts)

# Ubah hasil menjadi DataFrame pandas agar mudah dibaca
df_tfidf_default = pd.DataFrame(
    tfidf_default.toarray(),                                # ubah sparse matrix jadi array biasa
    columns=vectorizer_default.get_feature_names_out(),       # nama kolom = daftar term
    index=[f"D{i+1}" for i in range(len(documents))],        # nama baris = D1..D4
).round(4)                                                     # bulatkan 4 angka desimal

df_tfidf_default = df_tfidf_default[vocabulary]   # urutkan kolom agar konsisten dengan tabel sebelumnya

print("=== TF-IDF (scikit-learn default: smooth_idf=True, norm='l2') ===")
df_tfidf_default   # tampilkan tabel TF-IDF versi default (ternormalisasi)

=== TF-IDF (scikit-learn default: smooth_idf=True, norm='l2') ===


,antar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,efisien,ilmu,informasi,jaringan,kecerdasan,keras,komputer,kumpulan,lunak,manusia,membantu,membuat,mempelajari,memungkinkan,menemukan,mesin,pengguna,perangkat,pertukaran,relevan,saling,sistem,temu,terdiri,terhubung
D1,0.0000,0.0000,0.00,0.00,0.00,0.0000,0.00,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.00,0.3427,0.2187,0.0000,0.3427,0.00,0.0000,0.00,0.00,0.0000,0.0000,0.00,0.0000,0.5404,0.0000,0.0000,0.3427,0.2702,0.0000,0.3427,0.3427
D2,0.3615,0.0000,0.00,0.00,0.00,0.3615,0.00,0.2850,0.0000,0.3615,0.00,0.0000,0.3615,0.00,0.0000,0.2308,0.0000,0.0000,0.00,0.0000,0.00,0.00,0.3615,0.0000,0.00,0.0000,0.2850,0.3615,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,0.31,0.31,0.31,0.0000,0.31,0.0000,0.0000,0.0000,0.31,0.0000,0.0000,0.31,0.0000,0.1979,0.0000,0.0000,0.31,0.0000,0.31,0.31,0.0000,0.0000,0.31,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.3125,0.00,0.00,0.00,0.0000,0.00,0.2463,0.3125,0.0000,0.00,0.3125,0.0000,0.00,0.0000,0.0000,0.3125,0.0000,0.00,0.3125,0.00,0.00,0.0000,0.3125,0.00,0.3125,0.0000,0.0000,0.3125,0.0000,0.2463,0.3125,0.0000,0.0000


In [12]:
top_terms = []   # list kosong untuk menampung term dengan bobot tertinggi tiap dokumen

# Loop untuk setiap dokumen (D1, D2, D3, D4)
for doc_index in [f"D{i+1}" for i in range(len(documents))]:
    row = df_tfidf_manual.loc[doc_index]     # ambil satu baris (semua nilai TF-IDF) untuk dokumen ini
    top_term = row.idxmax()                   # cari nama term dengan nilai TF-IDF tertinggi pada baris ini

    # Simpan hasil (nama dokumen, term tertinggi, nilainya) sebagai satu baris data
    top_terms.append({
        "Dokumen": doc_index,
        "Term dengan TF-IDF Tertinggi": top_term,
        "Nilai TF-IDF": row[top_term],
    })

df_top = pd.DataFrame(top_terms)   # ubah list of dict menjadi DataFrame agar rapi
df_top   # tampilkan tabel ringkasan term tertinggi tiap dokumen

,Dokumen,Term dengan TF-IDF Tertinggi,Nilai TF-IDF
0,D1,perangkat,3.3863
1,D2,antar,2.3863
2,D3,buatan,2.3863
3,D4,besar,2.3863


**Analisis:**

Berdasarkan hasil perhitungan TF-IDF manual, setiap dokumen memiliki *term* dengan bobot tertinggi
yang cenderung merupakan kata kunci utama/topik dari dokumen tersebut (misalnya "komputer" atau
"perangkat" pada dokumen tentang sistem komputer, "jaringan" pada dokumen tentang jaringan komputer,
"kecerdasan" atau "cerdas" pada dokumen tentang AI, dan "temu"/"informasi" pada dokumen tentang
sistem temu kembali). Term tersebut penting karena memiliki frekuensi kemunculan (TF) yang cukup
tinggi pada dokumennya sendiri namun jarang muncul di dokumen lain (DF rendah, sehingga IDF-nya
tinggi). Kombinasi TF yang tinggi dan IDF yang tinggi inilah yang membuat term tersebut memiliki
daya pembeda (*discriminative power*) yang kuat — term seperti ini sangat berguna untuk membedakan
topik antar dokumen dan menjadi representasi kata kunci yang baik dalam sistem *temu kembali informasi*,
dibandingkan kata-kata umum yang muncul di banyak dokumen sehingga bobot IDF-nya rendah.